# Build complete curated Evidence Lab replays

30 examples: 10 per dataset, all from frozen Week-12 VAL. **New illustrative runs**, separate from the scientific study. Never access QASPER TEST or confirmation cohorts. All new files remain under `demo/`.

The complete matrix is **30 × 3 retrievers × 9 methods × 2 budgets = 1,620 answer rows**, plus a deletion run for every selected evidence unit. Exact prompts are reused. Publication is blocked until every combination and every unit-deletion result is complete.

Default Run All performs a plan and synthetic smoke only. To build, set `RUN_FULL=True` in the next cell and Run All. Restarting resumes completed shards. Leave all stage switches enabled for end-to-end operation.

In [ ]:
from pathlib import Path
import subprocess, json
ROOT = Path.cwd().resolve()
while not (ROOT/'demo/app.py').is_file():
    if ROOT == ROOT.parent: raise FileNotFoundError('Open within CS787-RAG-Project')
    ROOT = ROOT.parent
RUN_SMOKE = False
RUN_FULL = False
RESUME = True
OVERWRITE = False
RUN_PREPARATION = True
RUN_ANSWERS = True
RUN_ATTRIBUTION = True
RUN_PUBLICATION = True
if not RESUME or OVERWRITE: raise ValueError('Immutable results: keep RESUME=True and OVERWRITE=False')
def stage(name, full=False):
    cmd = [str(ROOT/'sara_env/bin/python'), str(ROOT/'demo/build_curated_replays.py'), name]
    if full: cmd.append('--full')
    subprocess.run(cmd, cwd=ROOT, check=True)


## 1 — Freeze the curated IDs and print the complete plan
Membership comes from the existing outcome-curated illustration catalog, filtered against frozen VAL IDs. No new winner selection or benchmark claim. Existing source/config/freeze hashes are recorded. Repeated execution must match this exact manifest; use a new output version for incompatible changes.

In [ ]:
stage('plan')


## 2 — Synthetic-only smoke
Tests run with fake evidence and answers in temporary directories. No model inference, downloads, historical result changes or new scientific scores.

In [ ]:
if RUN_SMOKE: stage('smoke')


## 3 — Retrieve and prepare ALL evidence variants
SFR, BM25 and MiniLM; passage, sentence, fixed hierarchy, adaptive QBGF, C2, no evidence, exact propositions and two/four-sentence-per-passage illustrations. Core methods call the frozen canonical implementations. Additional representation policies are labelled demo-only. All source offsets, top-10 ranks/scores, fine units, choices and exact evidence IDs are retained. SFR exits before Qwen starts.

In [ ]:
if RUN_FULL and RUN_PREPARATION: stage('prepare', full=True)


## 4 — Generate answers with the frozen Qwen BASE_FP16 reader
Same pinned model, prompt/parser, greedy decoding and budgets as the study. Each combination gets its own result, reusing only exactly identical query/prompt evidence IDs. Cost accounting remains conceptual, including cached calls. No evidence is fabricated.

In [ ]:
if RUN_FULL and RUN_ANSWERS: stage('answers', full=True)


## 5 — Complete the utility explorer
For each selected unit, remove only that unit, rebuild the exact evidence IDs, and generate/evaluate again. HELP/HURT/NEUTRAL comes from actual observed deletion effect. This can require many more generations than the 1,620 answer rows. The plan prints the exact deletion count after preparation; pause safely if needed. Every completed deletion is resumable. NONE has zero units and needs no deletion run.

In [ ]:
stage('plan')
if RUN_FULL and RUN_ATTRIBUTION: stage('attribution', full=True)


## 6 — Audit and publish one self-contained portable collection
Only publish after all answer and deletion rows exist. The compact bundle contains the questions, reference answers, all source hierarchies, retrieval pools, contexts, actual outputs, metrics and deletion results. It works even if intermediate embedding/preparation caches are later removed. Keep `bundle.json`, `PUBLISHED.json` and `manifest.json` backed up; they are the durable app data, not disposable caches.

In [ ]:
if RUN_FULL and RUN_PUBLICATION: stage('publish', full=True)
stage('plan')
if not RUN_FULL: print('Safe mode: no real model work started. Set RUN_FULL = False and Run All to build the complete collection.')


## 7 — Open the application
Restart Streamlit after publication. Evidence Lab automatically prefers **Complete curated demo**; historical research findings and the historical export view remain separately labelled.

`demo/.venv/bin/python -m streamlit run demo/app.py`

Do not delete `demo/replay_data/v1/bundle.json` or its publication receipt. Back up `demo/replay_data/v1/` separately while the whole application is ignored by git.